**Name:** Saad Abdulaziz Alshehri  
**Student ID:** 2240006495

# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [2]:
!pip -q install transformers sentencepiece torch


In [1]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [2]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [4]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [5]:
# task 1: choose the best model family for each task
answers = {
    "1. Sentiment classification": "Encoder-only (BERT / DistilBERT)",
    "2. Machine translation": "Encoder-decoder (T5 / BART)",
    "3. Open-ended text generation": "Decoder-only (GPT)",
    "4. Named Entity Recognition": "Encoder-only (BERT / RoBERTa)",
    "5. Text summarization": "Encoder-decoder (T5 / BART)",
}

for task, family in answers.items():
    print(task, "->", family)

1. Sentiment classification -> Encoder-only (BERT / DistilBERT)
2. Machine translation -> Encoder-decoder (T5 / BART)
3. Open-ended text generation -> Decoder-only (GPT)
4. Named Entity Recognition -> Encoder-only (BERT / RoBERTa)
5. Text summarization -> Encoder-decoder (T5 / BART)


**Explanation:**

- **Sentiment classification** and **NER** need the model to *understand* the whole input and label it (or label each token), so an **encoder-only** model fits best.
- **Machine translation** and **summarization** take one text as input and produce a different text as output, so we need an **encoder-decoder** model (the encoder reads, the decoder writes).
- **Open-ended text generation** only needs to continue the text word by word, so a **decoder-only** model (GPT) is the right choice.

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [6]:
# task 2: use the sentiment classifier on the three sentences
# (the classifier was already loaded in Part C)
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

for sentence in sentences:
    result = classifier(sentence)[0]
    print("Sentence:", sentence)
    print("Label:", result["label"], "| Score:", round(result["score"], 4))
    print()

Sentence: The movie was excellent.
Label: POSITIVE | Score: 0.9918

Sentence: The lecture was difficult to understand.
Label: NEGATIVE | Score: 0.9997

Sentence: I enjoyed learning about transformers.
Label: POSITIVE | Score: 0.9975



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [7]:
# task 3: summarize a short paragraph with FLAN-T5
# (tokenizer and model were already loaded in Part B)
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers "
    "understand human language. It is used in many applications such as translation, "
    "chatbots, search engines and sentiment analysis. Modern NLP systems are mostly based "
    "on transformer models, which can learn the meaning of words from the context around them."
)

prompt = "Summarize this text: " + paragraph

# convert text into tokens
inputs = tokenizer(prompt, return_tensors="pt")

# generate the summary
summary_ids = model.generate(**inputs, max_new_tokens=60)

# convert tokens back to text
print("Original paragraph:")
print(paragraph)
print()
print("Summary:")
print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

Original paragraph:
Natural language processing is a field of artificial intelligence that helps computers understand human language. It is used in many applications such as translation, chatbots, search engines and sentiment analysis. Modern NLP systems are mostly based on transformer models, which can learn the meaning of words from the context around them.

Summary:
Computers use natural language processing to learn the meaning of words.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


**Answer:**

A decoder-only model is suitable for a chatbot like ChatGPT because a chatbot is basically a text generation task: it reads the conversation so far and predicts the next token, then the next one, until the reply is finished. The decoder-only model is trained exactly for this (next token prediction), so it generates fluent and natural answers. It also uses one single model for both the input (the user's question) and the output (the reply), which makes it simple and easy to scale to very large sizes. Because of this, it can handle many different tasks (answering, translating, summarizing) just by changing the prompt.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [8]:
# optional challenge: same prompt, sampling instead of greedy decoding
prompt = "Natural language processing is"

# with a fixed seed -> same output every time
for i in range(3):
    set_seed(42)
    out = generator(prompt, max_new_tokens=20, do_sample=True)
    print("With seed", i + 1, ":", out[0]["generated_text"].replace("\n", " "))

print()

# without the seed -> different output every run
for i in range(3):
    out = generator(prompt, max_new_tokens=20, do_sample=True)
    print("No seed", i + 1, ":", out[0]["generated_text"].replace("\n", " "))

With seed 1 : Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words
With seed 2 : Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words
With seed 3 : Natural language processing is the most common language used by professional software developers. A language processing program that can process thousands of words

No seed 1 : Natural language processing is an important part of the overall English-speaking language. It is a complex learning process which can arise
No seed 2 : Natural language processing is a crucial part of many languages. Many languages are thought of as a function of the word order.
No seed 3 : Natural language processing is now the foundation of the modern language processing process. It’s a necessary part of the human


**Answer:**

When `do_sample=True`, the model does not always pick the most probable next token. It looks at the probability distribution of the next token and **randomly samples** from it. So the same prompt can give a different word at the first step, and then the rest of the sentence changes too. When we use `set_seed(42)` the random generator starts from the same point every time, so we get the same output. When we remove it, the random numbers are different on each run, so the output is different. (With `do_sample=False`, the model is greedy and always gives the same result.)